# Module 09 Lab: Building a Simple AI Agent with Computer Vision

Welcome to the exciting world of AI Agents! In this lab, you will build a simple **visual monitoring agent** that can watch a video and detect specific events. This is a foundational step towards creating intelligent systems that can perceive, reason, and act in the real world.

**What you'll learn:**
- The basics of AI agents and the Perception-Reasoning-Action (PRA) loop.
- How to use computer vision as the 'perception' part of an agent.
- How to build a simple agent that can detect and respond to events in a video.

## Learning Objectives

By the end of this lab, you will be able to:

- **Explain** the Perception-Reasoning-Action (PRA) loop.
- **Implement** a simple AI agent that uses a pre-trained object detection model to perceive its environment.
- **Design** a simple reasoning system to make decisions based on visual input.
- **Create** an agent that can take actions based on its reasoning.

## 1. Setup and Installation

We'll need a few libraries for this lab. We'll use `opencv-python` for video processing and `ultralytics` for the YOLO object detection model.

### What are these libraries?
- **ultralytics**: Provides a powerful and easy-to-use implementation of the YOLO (You Only Look Once) object detection model.
- **opencv-python**: A library for computer vision tasks, including reading and displaying video.
- **requests**: For downloading the video file.

In [ ]:
%pip install ultralytics opencv-python requests

Now, let's import the necessary components.

In [ ]:
from ultralytics import YOLO
import cv2
import requests
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

## 2. Understanding AI Agents and the PRA Loop

### What is an AI Agent?

An AI agent is a system that can perceive its environment, reason about what it perceives, and take actions to achieve its goals. A self-driving car is a great example of a complex AI agent.

### The Perception-Reasoning-Action (PRA) Loop

The PRA loop is the fundamental cycle that an AI agent follows:

1.  **Perception:** The agent gathers information about its environment. For our agent, this will be done using a computer vision model to 'see' the world through a video.
2.  **Reasoning:** The agent processes the information it has gathered and decides what to do. Our agent's reasoning will be simple: 'If I see a person, then I should raise an alert.'
3.  **Action:** The agent performs an action based on its reasoning. Our agent's action will be to print a message to the screen.

## Part 1: Coded Demonstration

In this part, we will build a simple visual monitoring agent that watches a video of a street and detects when a person appears.

In [ ]:
# 1. Load the YOLO model
# We are using the 'yolov8n.pt' model, which is a small and fast model.
model = YOLO('yolov8n.pt')

In [ ]:
# 2. Download a video to analyze
video_url = 'https://archive.org/download/popeye_shuteye_popeye/popeye_shuteye_popeye_512kb.mp4'
video_path = 'popeye.mp4'
r = requests.get(video_url, stream=True)
with open(video_path, 'wb') as f:
    for chunk in r.iter_content(chunk_size=1024*1024):
        if chunk:
            f.write(chunk)

In [ ]:
# 3. Create the AI Agent

def visual_monitoring_agent(video_path):
    """A simple agent that monitors a video for people."""

    cap = cv2.VideoCapture(video_path)
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        # Process every 10th frame to save time
        if frame_count % 10 != 0:
            continue

        # PERCEPTION: Use the YOLO model to detect objects in the frame
        results = model(frame)

        # REASONING: Check if a person was detected
        person_detected = False
        for result in results:
            for box in result.boxes:
                class_id = int(box.cls[0])
                # The class ID for 'person' in the COCO dataset is 0
                if class_id == 0:
                    person_detected = True
                    break
            if person_detected:
                break

        # ACTION: If a person was detected, print an alert
        if person_detected:
            print(f'Frame {frame_count}: Person detected!')

            # Optional: Display the frame with the detection
            annotated_frame = results[0].plot()
            img = Image.fromarray(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
            plt.imshow(img)
            plt.show()

    cap.release()

# Run the agent
visual_monitoring_agent(video_path)

## Part 2: Student Challenge

Now it's your turn! Your challenge is to modify the agent to monitor for a different object. You can choose any object that the YOLO model can detect (e.g., 'car', 'dog', 'cat', 'boat').

**Hint:** You will need to find the class ID for the object you want to detect. You can find a list of all COCO class IDs online (a quick search for 'COCO dataset classes' will help).

In [ ]:
# --- ENTER YOUR CODE HERE ---

# 1. Choose a new object to detect: Car - ID: 2
import requests
import cv2
from PIL import Image
import matplotlib.pyplot as plt

# Download a reliable sample video of street traffic
video_url = 'https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4'
video_path = 'car_sample.mp4'

print("Downloading video...")
r = requests.get(video_url, stream=True)
with open(video_path, 'wb') as f:
    for chunk in r.iter_content(chunk_size=1024*1024):
        if chunk:
            f.write(chunk)
print("Download complete.")

In [ ]:
# 2. Create a new agent function that detects your chosen object.
def car_monitoring_agent(video_path):
    """A simple agent that monitors a video for cars."""
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    cars_found_total = 0

    print(f"Starting analysis on {video_path}...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        # Process every 5th frame to save time
        if frame_count % 5 != 0:
            continue

        # PERCEPTION: Use the YOLO model to detect objects in the frame
        results = model(frame)

        # REASONING: Check if a car was detected (class ID 2)
        car_detected = False
        for result in results:
            for box in result.boxes:
                class_id = int(box.cls[0])
                if class_id == 2:  # 2 is the COCO class ID for 'car'
                    car_detected = True
                    cars_found_total += 1
                    break
            if car_detected:
                break

        # ACTION: If a car was detected, print an alert
        if car_detected:
            print(f'Frame {frame_count}: Car detected!')

            # Optional: Display the frame with the detection
            annotated_frame = results[0].plot()
            img = Image.fromarray(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
            plt.figure(figsize=(6, 4))
            plt.imshow(img)
            plt.axis('off')
            plt.show()

    cap.release()
    print(f"\nAnalysis complete! Processed {frame_count} total frames.")
    if cars_found_total == 0:
        print("No cars were detected in this video by the model.")

    else:
        print(f"{cars_found_total} cars were detected in total.")

# 3. Run your new agent on the video.
car_monitoring_agent(video_path)


## Reflective Questions

Please answer the following questions in a new Markdown cell below.

1.  In your own words, what are the three parts of the Perception-Reasoning-Action (PRA) loop? How did our agent implement each part?

- The 3 parts of the PRA are Perceptiom, Reasoning, and Action. The agents, for both the human and car detectors, whenever the agent looks for said obgects within each frame, it gathers the information within the image, determining what makes up or looks like the said object. For reasoning, the agent processes the gathered information to make decicions or predictions about the environment. As for action, the agent takes action based on its reasoning that what it is looking at is indeed what it is searching for.

2.  Our agent's reasoning was very simple ('if person, then alert'). Describe a more complex reasoning rule you could add to the agent. For example, what if you only wanted to detect people in a certain part of the screen?

- A more complex reasoning rule could involve checking the bounding box coordinates of the detected object to ensure it falls within a specific 'Region of Interest' (ROI), such as only alerting if a car enters a specific lane or a person enters a restricted zone. Another example is temporal reasoning, where the agent only triggers an action if the object is detected consistently across multiple consecutive frames to reduce false alarms.

3.  What are some of the challenges you might face when trying to use an agent like this in the real world? (e.g., lighting changes, different camera angles, etc.)

- Issues that may arise using it in real world scenerios, are changes in lighting, camera angles, fog, image quality, distortions, and ovoerload of data

4.  Describe a real-world application for a visual monitoring agent. What would it perceive, what reasoning would it use, and what actions would it take?

- A real-world application could be a smart automated safety system for a warehouse. It would PERCEIVE by using cameras to monitor the warehouse floor and detect objects like people and forklifts. It would use REASONING to check if the bounding boxes of a person and a hazardous 'forklift only' zone overlap. Its ACTION would be triggering a flashing warning light or sounding an alarm to prevent an accident.

## Submission Instructions

1.  Complete the **Student Challenge** section with your modified agent.
2.  Answer the **Reflective Questions** in a new Markdown cell.
3.  Save your completed notebook (`.ipynb` file) and submit it.